In [ ]:
"""
DBLP-ACM Ground Truth Generation Script
Generates consecutive adjacent matching pairs for records sharing a cluster ID.
Produces the definitive ground truth CSV.
"""

import os
from pathlib import Path
import pandas as pd

project_root = Path("").resolve()
os.chdir(project_root)

# ----------------------------
# Paths Configuration
# ----------------------------
SAMPLE_CSV_PATH = Path("./dataset/sample_DBLP_ACM.csv")
OUTPUT_GT_CSV = Path("./dataset/sample_DBLP_ACM_gt.csv")

# ----------------------------
# Core Functionality
# ----------------------------
def generate_ground_truth_adjacent_pairs(sample_csv_path: Path, output_csv_path: Path) -> pd.DataFrame:
    """
    Reads the sample dataset and pairs adjacent rows within each cluster 
    as positive matching instances.

    Expected columns in output: ['ltable_id', 'rtable_id']
    """
    print(f"Loading sampled data from {sample_csv_path}...")
    df = pd.read_csv(sample_csv_path)

    # Validate essential columns exist
    required_cols = {"id", "cluster_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in dataset: {missing}")

    # SAFEGUARD: Convert ID column to string/text to prevent pandas
    # from inferring numeric types, matching text ID requirements.
    df["id"] = df["id"].astype(str).str.strip()

    # Track original row order natively before groupby breaks it
    df = df.reset_index(drop=False).rename(columns={"index": "_row_order"})

    gt_rows = []

    # Iterate over unique clusters
    for cluster_id, group in df.groupby("cluster_id", sort=False):
        # Keep original row order to create sequential pairs
        group = group.sort_values("_row_order")
        ids = group["id"].tolist()

        # Connect elements sequentially (n-1 pairs for n cluster items)
        if len(ids) >= 2:
            for i in range(len(ids) - 1):
                gt_rows.append({
                    "ltable_id": ids[i],
                    "rtable_id": ids[i + 1]
                })

    # Materialize dataframe and write output
    gt_df = pd.DataFrame(gt_rows, columns=["ltable_id", "rtable_id"])
    gt_df.to_csv(output_csv_path, index=False)

    print(f"Saved Ground Truth File: {output_csv_path}")
    print(f"Total positive matching pairs generated: {len(gt_df)}")

    return gt_df

# ----------------------------
# Execution
# ----------------------------
if __name__ == "__main__":
    if SAMPLE_CSV_PATH.exists():
        gt = generate_ground_truth_adjacent_pairs(SAMPLE_CSV_PATH, OUTPUT_GT_CSV)

        print("\nGround Truth Preview:")
        print(gt.head(10))
    else:
        print(f"Error: Could not locate '{SAMPLE_CSV_PATH}'. Make sure the file is in the same directory.")